# Phase 6C — Final Test Evaluation

Single Colab notebook for **final Dec 16–22 test scoring** of configs locked in Phase 6B.

## Locked configurations (Phase 6B winners)

| Family | ID | Configuration |
|--------|----|---------------|
| SARIMA | A | `(1,0,1)×(1,0,1,144)` |
| LSTM | B | 64 units / 1 layer / dropout 0.1 / lr `1e-3` |
| TCN | D | filters 32 / kernel 5 / dilations `(1,2,4,8,16)` / lr `5e-4` |

## Protocol

- Squares: **5161 / 5059 / 5259**
- Sequence length **L = 144** (1 day at 10-min resolution)
- **SARIMA:** full-train fit (original scale), `maxiter=50`
- **LSTM / TCN:** train-only MinMax; early stopping on validation
- **Dec 16–22 OPEN for scoring** (test week)
- **No hyperparameter search on test** — configs are locked
- Cross-family ranking on test metrics is reporting only

## Google Drive layout

- Data → `milan_traffic/data/`
- Locked configs → `milan_traffic/results/phase6b/locked_configs.json`
- Outputs → `milan_traffic/results/phase6c/`


In [ ]:
!pip install -q statsmodels scikit-learn pandas numpy matplotlib


## 1 · Colab setup — mount Drive & set paths


In [ ]:
# Google Drive mount (Colab only)
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/milan_traffic"
else:
    PROJECT_ROOT = r"G:\My Drive\milan_traffic"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
LOCKED_DIR = os.path.join(PROJECT_ROOT, "results", "phase6b")
RESULTS = os.path.join(PROJECT_ROOT, "results", "phase6c")
os.makedirs(RESULTS, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("LOCKED_DIR  :", LOCKED_DIR)
print("RESULTS     :", RESULTS)
print(
    "data files  :",
    sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else "(missing — upload Phase 5 CSV)",
)
print(
    "phase6b     :",
    sorted(os.listdir(LOCKED_DIR)) if os.path.isdir(LOCKED_DIR) else "(missing — run Phase 6B first)",
)


## 2 · Imports & protocol constants


In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Input, Conv1D, Activation,
    SpatialDropout1D, Add, Lambda,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

# Protocol constants (match Phase 5 / 6B)
SQUARES = [5161, 5059, 5259]
SEQ_LEN = 144
HORIZON = 1

TRAIN_END = pd.Timestamp("2013-12-08 23:50", tz="UTC")
VAL_END = pd.Timestamp("2013-12-15 23:50", tz="UTC")
TEST_START = pd.Timestamp("2013-12-16 00:00", tz="UTC")
TEST_END = pd.Timestamp("2013-12-22 23:50", tz="UTC")

SARIMA_MAXITER = 50
EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 64

print("SQUARES:", SQUARES)
print("SEQ_LEN:", SEQ_LEN)
print("TEST:", TEST_START, "→", TEST_END)
print("SARIMA_MAXITER:", SARIMA_MAXITER, "| EPOCHS:", EPOCHS, "| PATIENCE:", PATIENCE)


## 3 · Data utilities, metrics & locked configs


In [ ]:
def _ensure_utc_index(series):
    # Make DatetimeIndex UTC-aware for comparison with protocol cutoffs.
    idx = pd.DatetimeIndex(series.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")
    out = series.copy()
    out.index = idx
    return out


def load_square(square_id, data_dir):
    # Prefer per-square CSV; skip empty placeholders; fall back to combo CSV.
    for fname in (f"square_{square_id}.csv", f"{square_id}.csv"):
        path = os.path.join(data_dir, fname)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            df = pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")
            return _ensure_utc_index(df["internet_traffic"].astype(float))

    combo = os.path.join(data_dir, "forecasting_target_squares.csv")
    if os.path.isfile(combo) and os.path.getsize(combo) > 0:
        df = pd.read_csv(combo, parse_dates=["timestamp"])
        sub = df.loc[df["square_id"].astype(int) == int(square_id)].copy()
        if sub.empty:
            raise FileNotFoundError(f"square_id {square_id} missing in {combo}")
        sub = sub.sort_values("timestamp").set_index("timestamp")
        return _ensure_utc_index(sub["internet_traffic"].astype(float))

    raise FileNotFoundError(
        f"Square {square_id} not found in {data_dir}. "
        f"Expected square_{square_id}.csv or forecasting_target_squares.csv"
    )


def split_tvt(series):
    # Return (train, val, test) as numpy arrays on original scale.
    train = series[series.index <= TRAIN_END].values
    val = series[(series.index > TRAIN_END) & (series.index <= VAL_END)].values
    test = series[(series.index >= TEST_START) & (series.index <= TEST_END)].values
    assert len(train) > 0, "Train split is empty"
    assert len(val) > 0, "Val split is empty"
    assert len(test) > 0, "Test split is empty"
    return train, val, test


def compute_metrics(y_true, y_pred, train_mean, label=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.maximum(np.abs(y_true), 1e-8)
    mape = float(np.mean(np.abs(y_true - y_pred) / denom) * 100)
    nmae = mae / train_mean
    nrmse = rmse / train_mean
    if label:
        print(
            f"  {label:30s}  MAE={mae:,.0f}  RMSE={rmse:,.0f}  "
            f"MAPE={mape:.2f}%  nMAE={nmae:.4f}  nRMSE={nrmse:.4f}"
        )
    return {"mae": mae, "rmse": rmse, "mape": mape, "nmae": nmae, "nrmse": nrmse}


def make_scaler(train):
    scaler = MinMaxScaler()
    tr_sc = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    return scaler, tr_sc


def inv(arr, scaler):
    return scaler.inverse_transform(np.array(arr).reshape(-1, 1)).flatten()


def make_train_sequences(train_sc, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(len(train_sc) - seq_len):
        X.append(train_sc[i : i + seq_len])
        y.append(train_sc[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def make_val_sequences(train, val_scaled, scaler, seq_len=SEQ_LEN):
    # First SEQ_LEN context steps from end of train (re-scaled).
    train_sc = scaler.transform(train.reshape(-1, 1)).flatten()
    context = np.concatenate([train_sc[-seq_len:], val_scaled])
    X, y = [], []
    for i in range(len(val_scaled)):
        X.append(context[i : i + seq_len])
        y.append(context[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def make_target_sequences(train, val, target, scaler, seq_len=SEQ_LEN):
    # Build one-step test windows; context from end of train+val history.
    train_sc = scaler.transform(train.reshape(-1, 1)).flatten()
    val_sc = scaler.transform(val.reshape(-1, 1)).flatten()
    target_sc = scaler.transform(target.reshape(-1, 1)).flatten()
    hist_sc = np.concatenate([train_sc, val_sc])
    context = np.concatenate([hist_sc[-seq_len:], target_sc])
    X, y = [], []
    for i in range(len(target_sc)):
        X.append(context[i : i + seq_len])
        y.append(context[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def plot_test(y_true, preds_dict, square_id, model_name, results_dir=None):
    plt.figure(figsize=(14, 4))
    n = len(y_true)
    x = np.arange(n)
    plt.plot(x, y_true, label="Actual (test)", color="black", linewidth=1.2)
    colors = ["steelblue", "tomato", "seagreen", "darkorange", "purple", "brown"]
    for (lbl, pred), col in zip(preds_dict.items(), colors):
        pred = np.asarray(pred, dtype=float).reshape(-1)
        if len(pred) < n:
            pred = np.concatenate([np.full(n - len(pred), np.nan), pred])
        elif len(pred) > n:
            pred = pred[:n]
        plt.plot(x, pred, label=lbl, alpha=0.8, linewidth=1.0, color=col)
    plt.title(f"Square {square_id} — {model_name} | Test week (Dec 16–22)")
    plt.xlabel("Step (10-min intervals)")
    plt.ylabel("Internet Traffic (bytes)")
    plt.legend()
    plt.tight_layout()
    if results_dir:
        path = os.path.join(results_dir, f"sq{square_id}_{model_name.lower()}_test.png")
        plt.savefig(path, dpi=120)
        print(f"  Plot saved → {path}")
    plt.show()


def load_locked_configs(locked_dir):
    # Accept Colab flat format OR nested models key; fallback = Phase 6B winners.
    defaults = {
        "SARIMA": {
            "candidate_id": "A",
            "order": [1, 0, 1],
            "seasonal_order": [1, 0, 1, 144],
        },
        "LSTM": {
            "candidate_id": "B",
            "units": 64,
            "n_layers": 1,
            "dropout": 0.1,
            "lr": 1e-3,
            "batch": 64,
        },
        "TCN": {
            "candidate_id": "D",
            "filters": 32,
            "kernel_size": 5,
            "dilations": [1, 2, 4, 8, 16],
            "dropout": 0.1,
            "lr": 5e-4,
            "batch": 64,
        },
    }
    path = os.path.join(locked_dir, "locked_configs.json")
    if not os.path.isfile(path):
        print(f"No locked_configs.json at {path} — using Phase 6B winner defaults")
        return defaults

    raw = json.load(open(path, encoding="utf-8-sig"))
    out = {}

    models = raw.get("models")
    if isinstance(models, dict) and models:
        for family, block in models.items():
            if not isinstance(block, dict):
                continue
            if block.get("status") == "unselected":
                continue
            cand = block.get("candidate_id") or block.get("label")
            cfg_block = dict(block.get("config") or {})
            for k, v in block.items():
                if k in ("candidate_id", "label", "status", "config", "validation_summary"):
                    continue
                cfg_block.setdefault(k, v)
            entry = {"candidate_id": str(cand) if cand is not None else None, **cfg_block}
            out[str(family).upper()] = entry
    else:
        for family in ("SARIMA", "LSTM", "TCN"):
            if family not in raw:
                continue
            block = dict(raw[family])
            cand = block.pop("label", block.pop("candidate_id", None))
            if "lr" in block and "learning_rate" not in block:
                block["learning_rate"] = block["lr"]
            if "batch" in block and "batch_size" not in block:
                block["batch_size"] = block["batch"]
            block.pop("rf", None)
            out[family] = {"candidate_id": str(cand) if cand is not None else None, **block}

    if not out:
        print(f"Empty configs in {path} — using defaults")
        return defaults

    for fam, dflt in defaults.items():
        out.setdefault(fam, dflt)
    print(f"Loaded locked configs from {path}")
    return out


LOCKED = load_locked_configs(LOCKED_DIR)
print("LOCKED:")
print(json.dumps(LOCKED, indent=2, default=str))


## 4 · Guard — confirm test week length (1008 bins)


In [ ]:
# Guard: each square must have the full Dec 16–22 test week (1008 x 10-min bins)
EXPECTED_TEST_LEN = 1008
for _sq in SQUARES:
    _s = load_square(_sq, DATA_DIR)
    _train, _val, _test = split_tvt(_s)
    assert len(_test) == EXPECTED_TEST_LEN, (
        f"Square {_sq}: test len={len(_test)} expected {EXPECTED_TEST_LEN}"
    )
    assert (_s.index[_s.index <= TRAIN_END] < TEST_START).all()
    assert (
        _s.index[(_s.index > TRAIN_END) & (_s.index <= VAL_END)] < TEST_START
    ).all()
    print(
        f"  Square {_sq}: train={len(_train)} val={len(_val)} test={len(_test)} OK"
    )
print("Guard passed: test week is Dec 16–22 (1008 bins) for all squares")


## 5 · Naive persistence — test week

One-step persistence: prediction at t+1 equals observed value at t, using observed history through the test window (not recursive multi-step).


In [ ]:
naive_rows = []

for sq in SQUARES:
    series = load_square(sq, DATA_DIR)
    train, val, test = split_tvt(series)
    train_mean = float(np.mean(train))

    # Persistence needs observed lags through test (hist = train+val+test)
    hist = np.concatenate([train, val, test])
    # y_pred for each test step = previous observed value in hist
    offset = len(train) + len(val)
    y_pred = hist[offset - 1 : offset - 1 + len(test)]
    assert len(y_pred) == len(test)

    m = compute_metrics(test, y_pred, train_mean, label=f"Naive sq{sq}")
    naive_rows.append({"model": "Naive", "candidate_id": "persistence", "square_id": sq, **m})
    plot_test(test, {"Naive": y_pred}, sq, "Naive", RESULTS)

naive_df = pd.DataFrame(naive_rows)
naive_path = os.path.join(RESULTS, "naive_test_table.csv")
naive_df.to_csv(naive_path, index=False)
print("Saved", naive_path)
display(naive_df)


## 6 · SARIMA — full-train fit, test score

Uses locked `LOCKED['SARIMA']` order / seasonal_order. Fit on **full train**, `maxiter=50`. Predict by extending through val+test and taking the test portion only.


In [ ]:
def fit_sarima(train_unscaled, order, seasonal_order, maxiter=SARIMA_MAXITER):
    # Fit SARIMAX on full train (unscaled). Returns filtered results for .extend().
    p, d, q = order
    P, D, Q, s = seasonal_order
    print(
        f"    Fitting SARIMA({p},{d},{q})({P},{D},{Q},{s}) "
        f"on {len(train_unscaled)} train obs, maxiter={maxiter}..."
    )
    t0 = time.time()
    model = SARIMAX(
        train_unscaled,
        order=tuple(order),
        seasonal_order=tuple(seasonal_order),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res_opt = model.fit(method="lbfgs", low_memory=True, maxiter=maxiter, disp=False)
    res = model.filter(res_opt.params)
    elapsed = time.time() - t0
    print(f"    Done in {elapsed:.1f}s | AIC={res.aic:.1f}")
    return res, elapsed


def sarima_predict_test(fit_result, val, test):
    # Extend Kalman filter through val+test; return one-step preds for test only.
    post = np.concatenate([val, test]).astype(float)
    extended = fit_result.extend(endog=post)
    # fittedvalues on the extended result = one-step predictions for post period
    preds_post = np.asarray(extended.fittedvalues, dtype=float).reshape(-1)
    y_pred = preds_post[-len(test) :]
    assert len(y_pred) == len(test)
    return y_pred


sarima_cfg = LOCKED["SARIMA"]
sarima_order = tuple(sarima_cfg["order"])
sarima_seasonal = tuple(sarima_cfg["seasonal_order"])
sarima_cand = sarima_cfg.get("candidate_id", "A")
print(f"Locked SARIMA-{sarima_cand}: order={sarima_order} seasonal={sarima_seasonal}")

sarima_rows = []
for sq in SQUARES:
    series = load_square(sq, DATA_DIR)
    train, val, test = split_tvt(series)
    train_mean = float(np.mean(train))

    res, fit_s = fit_sarima(train, sarima_order, sarima_seasonal, maxiter=SARIMA_MAXITER)
    y_pred = sarima_predict_test(res, val, test)
    m = compute_metrics(test, y_pred, train_mean, label=f"SARIMA-{sarima_cand} sq{sq}")
    sarima_rows.append(
        {
            "model": "SARIMA",
            "candidate_id": sarima_cand,
            "square_id": sq,
            "fit_time_s": fit_s,
            **m,
        }
    )
    plot_test(test, {f"SARIMA-{sarima_cand}": y_pred}, sq, "SARIMA", RESULTS)

sarima_df = pd.DataFrame(sarima_rows)
sarima_path = os.path.join(RESULTS, "sarima_test_table.csv")
sarima_df.to_csv(sarima_path, index=False)
print("Saved", sarima_path)
display(sarima_df)


## 7 · LSTM — locked config, test score

Train with early stopping on validation; score test via `make_target_sequences`. Supports `lr`/`learning_rate` and `batch`/`batch_size`.


In [ ]:
def build_lstm(units, n_layers, dropout, lr, seq_len=SEQ_LEN):
    model = Sequential()
    for i in range(n_layers):
        return_sequences = i < n_layers - 1
        if i == 0:
            model.add(
                LSTM(
                    units,
                    return_sequences=return_sequences,
                    input_shape=(seq_len, 1),
                )
            )
        else:
            model.add(LSTM(units, return_sequences=return_sequences))
        model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(optimizer=Adam(lr), loss="mse")
    return model


lstm_cfg = LOCKED["LSTM"]
lstm_cand = lstm_cfg.get("candidate_id", "B")
lstm_units = int(lstm_cfg["units"])
lstm_layers = int(lstm_cfg["n_layers"])
lstm_dropout = float(lstm_cfg["dropout"])
lstm_lr = float(lstm_cfg.get("lr", lstm_cfg.get("learning_rate", 1e-3)))
lstm_batch = int(lstm_cfg.get("batch", lstm_cfg.get("batch_size", BATCH_SIZE)))
print(
    f"Locked LSTM-{lstm_cand}: units={lstm_units} layers={lstm_layers} "
    f"dropout={lstm_dropout} lr={lstm_lr} batch={lstm_batch}"
)

lstm_rows = []
for sq in SQUARES:
    series = load_square(sq, DATA_DIR)
    train, val, test = split_tvt(series)
    train_mean = float(np.mean(train))

    scaler, train_sc = make_scaler(train)
    val_sc = scaler.transform(val.reshape(-1, 1)).flatten()
    X_tr, y_tr = make_train_sequences(train_sc)
    X_va, y_va = make_val_sequences(train, val_sc, scaler)
    X_te, _ = make_target_sequences(train, val, test, scaler)

    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    model = build_lstm(lstm_units, lstm_layers, lstm_dropout, lstm_lr)
    cb = [
        EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    ]
    t0 = time.time()
    model.fit(
        X_tr[..., np.newaxis],
        y_tr,
        validation_data=(X_va[..., np.newaxis], y_va),
        epochs=EPOCHS,
        batch_size=lstm_batch,
        callbacks=cb,
        verbose=0,
    )
    fit_s = time.time() - t0

    preds_sc = model.predict(X_te[..., np.newaxis], verbose=0).flatten()
    y_pred = inv(preds_sc, scaler)
    m = compute_metrics(test, y_pred, train_mean, label=f"LSTM-{lstm_cand} sq{sq}")
    lstm_rows.append(
        {
            "model": "LSTM",
            "candidate_id": lstm_cand,
            "square_id": sq,
            "fit_time_s": fit_s,
            **m,
        }
    )
    plot_test(test, {f"LSTM-{lstm_cand}": y_pred}, sq, "LSTM", RESULTS)
    del model
    tf.keras.backend.clear_session()

lstm_df = pd.DataFrame(lstm_rows)
lstm_path = os.path.join(RESULTS, "lstm_test_table.csv")
lstm_df.to_csv(lstm_path, index=False)
print("Saved", lstm_path)
display(lstm_df)


## 8 · TCN — locked config, test score

Causal residual TCN (same architecture as Phase 6B). Uses `LOCKED['TCN']`.


In [ ]:
def tcn_receptive_field(dilations, kernel_size):
    return 1 + 2 * (kernel_size - 1) * int(sum(dilations))


def residual_block(x, filters, kernel_size, dilation, dropout_rate):
    # Causal dilated Conv -> ReLU -> SpatialDropout (x2) + residual
    h = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation,
        kernel_initializer="he_normal",
    )(x)
    h = Activation("relu")(h)
    h = SpatialDropout1D(dropout_rate)(h)
    h = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation,
        kernel_initializer="he_normal",
    )(h)
    h = Activation("relu")(h)
    h = SpatialDropout1D(dropout_rate)(h)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1, padding="same")(x)
    return Activation("relu")(Add()([x, h]))


def build_tcn(filters, kernel_size, dilations, dropout, lr, seq_len=SEQ_LEN):
    rf = tcn_receptive_field(dilations, kernel_size)
    print(f"    TCN RF={rf} steps (SEQ_LEN={seq_len})")
    inp = Input(shape=(seq_len, 1))
    x = inp
    for d in dilations:
        x = residual_block(x, filters, kernel_size, dilation=d, dropout_rate=dropout)
    x = Lambda(lambda t: t[:, -1, :])(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(lr), loss="mse")
    return model


tcn_cfg = LOCKED["TCN"]
tcn_cand = tcn_cfg.get("candidate_id", "D")
tcn_filters = int(tcn_cfg["filters"])
tcn_kernel = int(tcn_cfg["kernel_size"])
tcn_dilations = tuple(tcn_cfg["dilations"])
tcn_dropout = float(tcn_cfg["dropout"])
tcn_lr = float(tcn_cfg.get("lr", tcn_cfg.get("learning_rate", 5e-4)))
tcn_batch = int(tcn_cfg.get("batch", tcn_cfg.get("batch_size", BATCH_SIZE)))
print(
    f"Locked TCN-{tcn_cand}: f={tcn_filters} k={tcn_kernel} "
    f"dil={tcn_dilations} dropout={tcn_dropout} lr={tcn_lr} batch={tcn_batch}"
)

tcn_rows = []
for sq in SQUARES:
    series = load_square(sq, DATA_DIR)
    train, val, test = split_tvt(series)
    train_mean = float(np.mean(train))

    scaler, train_sc = make_scaler(train)
    val_sc = scaler.transform(val.reshape(-1, 1)).flatten()
    X_tr, y_tr = make_train_sequences(train_sc)
    X_va, y_va = make_val_sequences(train, val_sc, scaler)
    X_te, _ = make_target_sequences(train, val, test, scaler)

    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    model = build_tcn(tcn_filters, tcn_kernel, tcn_dilations, tcn_dropout, tcn_lr)
    cb = [
        EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    ]
    t0 = time.time()
    model.fit(
        X_tr[..., np.newaxis],
        y_tr,
        validation_data=(X_va[..., np.newaxis], y_va),
        epochs=EPOCHS,
        batch_size=tcn_batch,
        callbacks=cb,
        verbose=0,
    )
    fit_s = time.time() - t0

    preds_sc = model.predict(X_te[..., np.newaxis], verbose=0).flatten()
    y_pred = inv(preds_sc, scaler)
    m = compute_metrics(test, y_pred, train_mean, label=f"TCN-{tcn_cand} sq{sq}")
    tcn_rows.append(
        {
            "model": "TCN",
            "candidate_id": tcn_cand,
            "square_id": sq,
            "fit_time_s": fit_s,
            **m,
        }
    )
    plot_test(test, {f"TCN-{tcn_cand}": y_pred}, sq, "TCN", RESULTS)
    del model
    tf.keras.backend.clear_session()

tcn_df = pd.DataFrame(tcn_rows)
tcn_path = os.path.join(RESULTS, "tcn_test_table.csv")
tcn_df.to_csv(tcn_path, index=False)
print("Saved", tcn_path)
display(tcn_df)


## 9 · Aggregate results

Concatenate all model rows, write `phase6c_square_results.csv`, summarise by mean nMAE, and emit `phase6c_comparison.json` with the best research model (reporting only — no HP search).


In [ ]:
# Ensure Naive has candidate_id
if "candidate_id" not in naive_df.columns:
    naive_df = naive_df.copy()
    naive_df["candidate_id"] = "persistence"

all_rows = pd.concat([naive_df, sarima_df, lstm_df, tcn_df], ignore_index=True)
square_path = os.path.join(RESULTS, "phase6c_square_results.csv")
all_rows.to_csv(square_path, index=False)
print("Saved", square_path)
display(all_rows)

rows_sum = []
for model, grp in all_rows.groupby("model", sort=False):
    cand = (
        grp["candidate_id"].dropna().iloc[0]
        if "candidate_id" in grp.columns and grp["candidate_id"].notna().any()
        else ("persistence" if model == "Naive" else "?")
    )
    rows_sum.append(
        {
            "model": model,
            "candidate_id": cand,
            "is_research_model": model != "Naive",
            "n_squares": int(grp["square_id"].nunique()),
            "mean_mae": float(grp["mae"].mean()),
            "mean_rmse": float(grp["rmse"].mean()),
            "mean_mape": float(grp["mape"].mean()),
            "mean_nmae": float(grp["nmae"].mean()),
            "mean_nrmse": float(grp["nrmse"].mean()),
        }
    )
summary = pd.DataFrame(rows_sum).sort_values("mean_nmae").reset_index(drop=True)
summary_path = os.path.join(RESULTS, "phase6c_summary.csv")
summary.to_csv(summary_path, index=False)
print("Saved", summary_path)
display(summary)

research = summary.loc[summary["is_research_model"]].copy()
best = research.iloc[0].to_dict() if not research.empty else None

comparison = {
    "phase": "6C",
    "test_week": "2013-12-16 → 2013-12-22",
    "squares": SQUARES,
    "locked_configs": LOCKED,
    "ranking_criterion": "mean_nmae (reporting only; configs locked in 6B)",
    "summary": summary.to_dict(orient="records"),
    "best_research_model": best,
}
comp_path = os.path.join(RESULTS, "phase6c_comparison.json")
with open(comp_path, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2, default=str)
print("Saved", comp_path)
if best:
    print(
        f"Best research model (by mean nMAE): {best['model']}-{best['candidate_id']} "
        f"mean_nMAE={best['mean_nmae']:.4f}"
    )
print("Phase 6C final evaluation done.")

